In [1]:
import sys
sys.path.append("../")

In [2]:
import torch
from mpsqsc.models import mps_classifier, mpstate
from importlib import reload

reload(mps_classifier)
reload(mpstate)

<module 'mpsqsc.models.mpstate' from '/Users/keisuke/Documents/projects/mps4qsc/notebooks/../mpsqsc/models/mpstate/__init__.py'>

In [3]:
import torch.nn.functional as F

L = 10
chi = 2
d = 2
qsc = mps_classifier.MpsQsc(L, chi, d)
allup = torch.zeros(L, d, dtype = torch.float64)
allup[:, 0] = 1.0
alldown = torch.zeros(L, d, dtype = torch.float64)
alldown[:, 1] = 1.0

mps_allup = mpstate.build_product_state(L, d, allup)
mps_alldown = mpstate.build_product_state(L, d, alldown)

mps_allup = mps_allup.normalize()
mps_alldown = mps_alldown.normalize()

mpsghz = mpstate.build_ghz_state(L, d, chi)
mpsghz = mpsghz.normalize()
qsc = mps_classifier.build_qsc_from_mpstate(mpsghz)

logits_ghz = qsc.contract_with_state(mpsghz)
logits_allup = qsc.contract_with_state(mps_allup)
logits_alldown = qsc.contract_with_state(mps_alldown)

F.softmax(logits_ghz, dim=-1), F.softmax(logits_allup, dim=-1)


(tensor([0.7311, 0.2689], dtype=torch.float64, grad_fn=<SoftmaxBackward0>),
 tensor([0.6698, 0.3302], dtype=torch.float64, grad_fn=<SoftmaxBackward0>))

In [ ]:
mpsghz = mpstate.build_ghz_state(L, d, chi)
mpsghz2 = mpstate.build_ghz_state(L, d, chi)
As = mpsghz2.As
As[0][:, 1] = -As[0][:, 1]
mpsghz2.set_As(As)
qsc = mps_classifier.build_2qsc_from_mpstate(mpsghz, mpsghz2)

In [9]:
qsc.As[0][:, :2]

tensor([[ 1., -0.],
        [ 0., -1.]], dtype=torch.float64, grad_fn=<SliceBackward0>)

In [5]:
qsc.contract_with_state(mpsghz)

tensor([2., 2.], dtype=torch.float64, grad_fn=<ViewBackward0>)

In [65]:
import random
from typing import List, Tuple, Dict
import torch
import torch.nn.functional as F

@torch.no_grad()
def _ensure_same_device_dtype(state, device, dtype):
    if getattr(state, "device", None) != device or getattr(state, "dtype", None) != dtype:
        state.to(device=device, dtype=dtype)
    return state

def train_classifier_qsc_on_ghz_vs_rho(
    qsc,
    mpsghz,
    mps_allup,
    mps_alldown,
    *,
    steps: int = 2000,
    lr: float = 1e-2,
    batch_size: int = 16,          # kept for API compatibility but ignored
    ghz_fraction: float = 0.5,     # ignored (fixed composition batch)
    weight_decay: float = 0.0,
    grad_clip: float | None = None,
    seed: int | None = 0,
    log_every: int = 100,
) -> Dict[str, List[float]]:
    """
    Train qsc so that it outputs class 1 for GHZ, class 0 for the mixture ρ.

    CHANGE: Each step uses a fixed batch of 4 samples:
        [GHZ, GHZ, all-up, all-down] with labels [1, 1, 0, 0].

    qsc.contract_with_state(state) -> logits (2,)
    CrossEntropyLoss uses logits directly.
    """
    if seed is not None:
        torch.manual_seed(seed)
        random.seed(seed)

    device, dtype = qsc.device, qsc.dtype

    # Ensure states match qsc's device/dtype
    _ensure_same_device_dtype(mpsghz, device, dtype)
    _ensure_same_device_dtype(mps_allup, device, dtype)
    _ensure_same_device_dtype(mps_alldown, device, dtype)

    # Optimizer over qsc parameters (tensors/Parameters with requires_grad=True)
    optim = torch.optim.Adam(qsc.parameters(), lr=lr, weight_decay=weight_decay)

    history = {"loss": [], "acc": []}

    # ---- fixed batch sampler ----
    def sample_batch() -> Tuple[List, torch.Tensor]:
        xs = [mpsghz, mpsghz, mps_allup, mps_alldown]
        y = torch.tensor([1, 1, 0, 0], dtype=torch.long, device=device)
        return xs, y

    for step in range(1, steps + 1):
        # Make this a no-op if qsc is not an nn.Module; otherwise toggles train mode.
        getattr(qsc, "train", lambda *args, **kwargs: None)()

        optim.zero_grad(set_to_none=True)

        # --- fixed mini-batch (size=4; 2 GHZ, 1 all-up, 1 all-down) ---
        states, y = sample_batch()

        # --- forward ---
        logits_list = []
        for st in states:
            logits = qsc.contract_with_state(st)         # (2,)
            logits_list.append(logits)
        logits_batch = torch.stack(logits_list, dim=0)   # (4, 2)
        print(logits_batch)

        # --- loss ---
        loss = F.cross_entropy(logits_batch, y)

        # --- backward ---
        loss.backward()

        # optional grad clipping
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(qsc.parameters(), max_norm=grad_clip)

        optim.step()

        # --- simple accuracy on this batch ---
        with torch.no_grad():
            probs = F.softmax(logits_batch, dim=-1)      # (4, 2)
            pred = probs.argmax(dim=-1)                  # (4,)
            acc = (pred == y).float().mean().item()
            # print(probs)  # uncomment if you want to see probabilities every step

        history["loss"].append(loss.item())
        history["acc"].append(acc)

        if (log_every is not None) and (step % log_every == 0):
            print(f"[step {step:5d}] loss={loss.item():.6f}  acc={acc:.3f}")

    return history


In [66]:
# You already created these:
# L = 10; chi = 2; d = 2
# qsc = mps_classifier.MpsQsc(L, chi, d)
# mps_allup, mps_alldown, mpsghz prepared and normalized
# qsc = mps_classifier.build_qsc_from_mpstate(mpsghz)   # if you prefer that init

hist = train_classifier_qsc_on_ghz_vs_rho(
    qsc,
    mpsghz=mpsghz,
    mps_allup=mps_allup,
    mps_alldown=mps_alldown,
    steps=150,
    lr=1e-2,
    weight_decay=0.0,
    grad_clip=1.0,   # optional
    seed=0,
    log_every=1,
)

# Quick check on the 3 canonical states:
with torch.no_grad():
    def probs_of(state):
        logits = qsc.contract_with_state(state)  # (2,)
        return F.softmax(logits, dim=-1).cpu()

    print("P(class | GHZ):     ", probs_of(mpsghz))       # want ~ [low, high]
    print("P(class | all-up):  ", probs_of(mps_allup))    # want ~ [high, low]
    print("P(class | all-down):", probs_of(mps_alldown))  # want ~ [high, low]


tensor([[1.0000, 0.0000],
        [1.0000, 0.0000],
        [0.7071, 0.0000],
        [0.7071, 0.0000]], dtype=torch.float64, grad_fn=<StackBackward0>)
[step     1] loss=0.857048  acc=0.500
tensor([[0.3648, 0.0037],
        [0.3648, 0.0037],
        [0.2579, 0.0026],
        [0.2579, 0.0026]], dtype=torch.float64, grad_fn=<StackBackward0>)
[step     2] loss=0.731751  acc=0.500
tensor([[0.1482, 0.0029],
        [0.1482, 0.0029],
        [0.1048, 0.0020],
        [0.1048, 0.0020]], dtype=torch.float64, grad_fn=<StackBackward0>)
[step     3] loss=0.705767  acc=0.500
tensor([[0.0690, 0.0019],
        [0.0690, 0.0019],
        [0.0488, 0.0013],
        [0.0488, 0.0013]], dtype=torch.float64, grad_fn=<StackBackward0>)
[step     4] loss=0.698488  acc=0.500
tensor([[0.0359, 0.0012],
        [0.0359, 0.0012],
        [0.0254, 0.0009],
        [0.0254, 0.0009]], dtype=torch.float64, grad_fn=<StackBackward0>)
[step     5] loss=0.695801  acc=0.500
tensor([[0.0204, 0.0008],
        [0.0204, 0.0008]

KeyboardInterrupt: 